# MongoDB CRUD Operations with Python (PyMongo)

## Learning objectives
By the end of this notebook, you will be able to:
- Connect to a MongoDB server running on your Windows machine.
- Create a database and collection.
- Insert one or many documents.
- Read and filter documents.
- Update existing documents.
- Delete documents.

**Prerequisite:** Install and start MongoDB Community Server on Windows. The default local MongoDB address is `mongodb://localhost:27017/`.

> MongoDB uses *documents* (similar to Python dictionaries) stored inside *collections*. Collections are similar to tables in relational databases.


## 1. Install the Python driver

PyMongo is the official Python library used in this notebook to communicate with MongoDB.

Run the following cell. If Jupyter asks you to restart the kernel after installation, do so and continue from the top.


In [1]:
%pip install pymongo

  Using cached dnspython-2.9.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/827.6 kB ? eta -:--:--
   ---------------------------------------- 827.6/827.6 kB 15.8 MB/s  0:00:00
Using cached dnspython-2.9.2-py3-none-any.whl (360 kB)

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ------

## 2. Import PyMongo and connect to MongoDB

`MongoClient` creates a connection to the MongoDB server.

- `localhost` means MongoDB is running on this same computer.
- `27017` is MongoDB's default port.
- `ping` checks whether the server responds.

If this cell reports a connection error, make sure the MongoDB Windows service is running.


In [2]:
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError, PyMongoError

client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=5000)

try:
    print(client.admin.command("ping"))
    print("MongoDB connection successful!")
except ServerSelectionTimeoutError:
    print("Could not connect. Start the MongoDB service and run this cell again.")

{'ok': 1.0}
MongoDB connection successful!


## 3. Select a database and collection

MongoDB creates a database and collection when data is first inserted.

- `college` is the database name.
- `students` is the collection name.
- `db` and `students` are Python variables that let us work with them.


In [3]:
db = client["college"]
students = db["students"]

print("Database:", db.name)
print("Collection:", students.name)

Database: college
Collection: students


## 4. CREATE — Insert documents

A MongoDB document is similar to a Python dictionary. It stores data as key-value pairs.

### Insert one document

`insert_one()` inserts one document. MongoDB automatically assigns it a unique `_id` if you do not provide one.


In [4]:
student = {
    "name": "Veera",
    "age": 30,
    "course": "Python"
}

result = students.insert_one(student)

print("Inserted document ID:", result.inserted_id)

Inserted document ID: 6b01c90e4561f05f85f37895


### Insert multiple documents

`insert_many()` inserts a list of documents in one operation. Run this cell once to add the sample records. If you run it again, it will insert duplicates because each insert is a new document.


In [8]:
more_students = [
    {"name": "Sai", "age": 25, "course": "Python"},
    {"name": "Ravi", "age": 28, "course": "AWS"},
    {"name": "Krishna", "age": 26, "course": "DevOps"}
]

result = students.insert_many(more_students)
print("Number of documents inserted:", len(result.inserted_ids))

Number of documents inserted: 3


In [9]:
more_students = [
    {"name": "Akki", "age": 9, "course": "Python"},
]

result = students.insert_many(more_students)
print("Number of documents inserted:", len(result.inserted_ids))

Number of documents inserted: 1


## 5. READ — Retrieve documents

### Read all documents

`find()` returns all documents that match the filter. An empty filter `{}` means “match everything”.


In [10]:
for student in students.find({}):
    print(student)

{'_id': ObjectId('6b01652c18df78f6cfe33ba1'), 'name': 'Veera', 'age': 31, 'course': 'Python and MongoDB', 'status': 'Active'}
{'_id': ObjectId('6b01653018df78f6cfe33ba3'), 'name': 'Ravi', 'age': 28, 'course': 'AWS', 'status': 'Active'}
{'_id': ObjectId('6b01653018df78f6cfe33ba4'), 'name': 'Krishna', 'age': 26, 'course': 'DevOps', 'status': 'Active'}
{'_id': ObjectId('6b016e7c18df78f6cfe33ba5'), 'name': 'Akki', 'age': 9, 'course': 'Python'}
{'_id': ObjectId('6b01c90e4561f05f85f37895'), 'name': 'Veera', 'age': 30, 'course': 'Python'}
{'_id': ObjectId('6b01c9114561f05f85f37896'), 'name': 'Sai', 'age': 25, 'course': 'Python'}
{'_id': ObjectId('6b01c9114561f05f85f37897'), 'name': 'Ravi', 'age': 28, 'course': 'AWS'}
{'_id': ObjectId('6b01c9114561f05f85f37898'), 'name': 'Krishna', 'age': 26, 'course': 'DevOps'}
{'_id': ObjectId('6b01c9134561f05f85f37899'), 'name': 'Akki', 'age': 9, 'course': 'Python'}
{'_id': ObjectId('6b01c9444561f05f85f3789a'), 'name': 'Sai', 'age': 25, 'course': 'Python'}


### Read one document

`find_one()` returns the first matching document, or `None` if no document matches.


In [11]:
student = students.find_one({"name": "Veera"})
print(student)

{'_id': ObjectId('6b01652c18df78f6cfe33ba1'), 'name': 'Veera', 'age': 31, 'course': 'Python and MongoDB', 'status': 'Active'}


### Filter documents

MongoDB query operators let you filter documents:
- `$gt`: greater than
- `$lt`: less than
- `$gte`: greater than or equal to
- `$lte`: less than or equal to

The following query finds students whose age is greater than 25.


In [12]:
for student in students.find({"age": {"$gt": 25}}):
    print(student)

{'_id': ObjectId('6b01652c18df78f6cfe33ba1'), 'name': 'Veera', 'age': 31, 'course': 'Python and MongoDB', 'status': 'Active'}
{'_id': ObjectId('6b01653018df78f6cfe33ba3'), 'name': 'Ravi', 'age': 28, 'course': 'AWS', 'status': 'Active'}
{'_id': ObjectId('6b01653018df78f6cfe33ba4'), 'name': 'Krishna', 'age': 26, 'course': 'DevOps', 'status': 'Active'}
{'_id': ObjectId('6b01c90e4561f05f85f37895'), 'name': 'Veera', 'age': 30, 'course': 'Python'}
{'_id': ObjectId('6b01c9114561f05f85f37897'), 'name': 'Ravi', 'age': 28, 'course': 'AWS'}
{'_id': ObjectId('6b01c9114561f05f85f37898'), 'name': 'Krishna', 'age': 26, 'course': 'DevOps'}
{'_id': ObjectId('6b01c9444561f05f85f3789b'), 'name': 'Ravi', 'age': 28, 'course': 'AWS'}
{'_id': ObjectId('6b01c9444561f05f85f3789c'), 'name': 'Krishna', 'age': 26, 'course': 'DevOps'}


### Select specific fields (projection)

The second argument to `find()` controls which fields are returned. Here, `_id` is excluded and only `name` and `course` are shown.


In [13]:
for student in students.find({}, {"_id": 0, "name": 1, "course": 1}):
    print(student)

{'name': 'Veera', 'course': 'Python and MongoDB'}
{'name': 'Ravi', 'course': 'AWS'}
{'name': 'Krishna', 'course': 'DevOps'}
{'name': 'Akki', 'course': 'Python'}
{'name': 'Veera', 'course': 'Python'}
{'name': 'Sai', 'course': 'Python'}
{'name': 'Ravi', 'course': 'AWS'}
{'name': 'Krishna', 'course': 'DevOps'}
{'name': 'Akki', 'course': 'Python'}
{'name': 'Sai', 'course': 'Python'}
{'name': 'Ravi', 'course': 'AWS'}
{'name': 'Krishna', 'course': 'DevOps'}
{'name': 'Akki', 'course': 'Python'}


## 6. UPDATE — Modify documents

`update_one(filter, update)` finds one matching document and changes it.

- The filter `{"name": "Veera"}` identifies the document.
- `$set` changes the specified field without replacing the whole document.
- `matched_count` tells how many documents matched.
- `modified_count` tells how many documents were actually changed.


In [14]:
result = students.update_one(
    {"name": "Veera"},
    {"$set": {"age": 31, "course": "Python and MongoDB"}}
)

print("Matched:", result.matched_count)
print("Modified:", result.modified_count)

Matched: 1
Modified: 0


### Update multiple documents

`update_many()` changes every document matching the filter. This example adds a `status` field to all students whose age is at least 25.


In [15]:
result = students.update_many(
    {"age": {"$gte": 25}},
    {"$set": {"status": "Active"}}
)

print("Matched:", result.matched_count)
print("Modified:", result.modified_count)

Matched: 10
Modified: 7


## 7. DELETE — Remove documents

**Be careful:** Delete operations permanently remove matching documents.

### Delete one document

`delete_one()` removes the first document matching the filter. Always use a specific filter when you intend to remove one record.


In [16]:
result = students.delete_one({"name": "Sai"})
print("Documents delete" \
"d:", result.deleted_count)

Documents deleted: 1


### Delete multiple documents

`delete_many()` removes all documents matching the filter. The example removes students with the course value `AWS`.

Avoid running `delete_many({})` unless you intentionally want to remove every document in the collection.


In [17]:
result = students.delete_many({"course": "AWS"})
print("Documents deleted:", result.deleted_count)

Documents deleted: 3


## 8. Verify the final collection

Run this cell to view the documents remaining after the operations above.


In [18]:
for student in students.find({}, {"_id": 0}):
    print(student)

{'name': 'Veera', 'age': 31, 'course': 'Python and MongoDB', 'status': 'Active'}
{'name': 'Krishna', 'age': 26, 'course': 'DevOps', 'status': 'Active'}
{'name': 'Akki', 'age': 9, 'course': 'Python'}
{'name': 'Veera', 'age': 30, 'course': 'Python', 'status': 'Active'}
{'name': 'Krishna', 'age': 26, 'course': 'DevOps', 'status': 'Active'}
{'name': 'Akki', 'age': 9, 'course': 'Python'}
{'name': 'Sai', 'age': 25, 'course': 'Python', 'status': 'Active'}
{'name': 'Krishna', 'age': 26, 'course': 'DevOps', 'status': 'Active'}
{'name': 'Akki', 'age': 9, 'course': 'Python'}


## 9. Count documents

`count_documents({})` counts all documents in the collection. You can also pass a filter to count only matching documents.


In [19]:
print("Total students:", students.count_documents({}))
print("Active students:", students.count_documents({"status": "Active"}))

Total students: 9
Active students: 6


## 10. Practice exercises

Try these yourself before checking the suggested tasks:

1. Insert a student named `Anil`, age `22`, course `Java`.
2. Display only students enrolled in `Python`.
3. Find students whose age is less than `27`.
4. Update `Anil`'s age to `23`.
5. Delete the student named `Anil`.

Use the CRUD methods demonstrated above. You can run the cells below to check your work.


In [20]:
# Exercise area — write your own code below



## 11. Close the connection

Close the MongoDB client when you have finished working with it. If you want to continue using MongoDB later in this notebook, run the connection cell again.


In [21]:
client.close()
print("MongoDB connection closed.")

MongoDB connection closed.


---
## Quick revision

| CRUD operation | PyMongo method | Purpose |
|---|---|---|
| Create | `insert_one()` / `insert_many()` | Add documents |
| Read | `find()` / `find_one()` | Retrieve documents |
| Update | `update_one()` / `update_many()` | Modify documents |
| Delete | `delete_one()` / `delete_many()` | Remove documents |

**Remember:** This notebook connects to a MongoDB server. Installing `pymongo` alone does not install or start MongoDB Server.
